In [33]:
from sklearn.cluster import MeanShift, estimate_bandwidth
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## Leitura dos arquivos de texto:

In [34]:
def read_csv(filename):
    df = pd.read_csv(filename)
    return df.values.tolist()

ibrahim_samples = read_csv("Datasets/Ibrahim-dataset/Entire-dataset/test_samples.csv")
ibrahim_labels = read_csv("Datasets/Ibrahim-dataset/Entire-dataset/test_labels.csv")


## Criação de um Dataframe unificando as samples e as labels:

In [35]:
df = pd.DataFrame(ibrahim_samples, columns=['h2','ch2','c2h2','c2h4','c2h6'])
df['act'] = np.array(ibrahim_labels).flatten()
print(df)

          h2      ch2    c2h2    c2h4    c2h6  act
0    123.000   50.700   65.90   62.00    9.00    4
1    392.000  153.000  236.00   45.00   82.00    4
2      0.001    0.001   35.00    5.00   10.00    4
3    169.000   38.000    5.80    6.50   48.50    3
4    400.000  940.000   24.00  820.00  210.00    7
..       ...      ...     ...     ...     ...  ...
513   32.730   16.030    0.52   38.43   50.62    1
514   59.980   78.020    0.31   34.94   65.89    1
515   80.650   65.570    0.93    1.57   25.43    1
516   90.720   66.180    0.17   34.79   28.00    1
517   74.380   69.360    0.16   23.75   19.85    1

[518 rows x 6 columns]


## Agrupando as amostras pelas falhas:

In [36]:
groups = df.groupby('act')
dict_groups = {name: group for name, group in groups}

print(dict_groups[4])

           h2       ch2    c2h2      c2h4     c2h6  act
0     123.000    50.700    65.9    62.000    9.000    4
1     392.000   153.000   236.0    45.000   82.000    4
2       0.001     0.001    35.0     5.000   10.000    4
9     629.000   402.000  1127.0   298.000   16.000    4
13     22.000    14.000    56.0     6.000   19.000    4
..        ...       ...     ...       ...      ...  ...
428   156.000    55.000    16.0    13.000  103.000    4
430    10.000    15.000    35.0     0.001    0.001    4
436  5100.000  1430.000  1010.0  1140.000    0.001    4
441    85.000    49.000   399.0    50.000    4.000    4
442    34.000    21.000    56.0    49.000    4.000    4

[119 rows x 6 columns]


## Função para ajustar o modelo e definir os centroides:

In [37]:
def mean_shift_generator(X_class, target_total):
    # Normalização para clustering mais estável
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_class)

    # Estima a largura de banda e aplica Mean Shift
    bandwidth = estimate_bandwidth(X_scaled, quantile=0.3, n_samples=min(500, len(X_scaled)))
    ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
    ms.fit(X_scaled)

    labels = ms.labels_
    synthetic_samples = []

    n_clusters = len(np.unique(labels))
    samples_per_cluster = max(1, target_total // n_clusters)

    for cluster_label in np.unique(labels):
        cluster_points = X_class[labels == cluster_label]  # usa os dados *originais* não normalizados

        if len(cluster_points) < 2:
            continue

        centroid = cluster_points.mean(axis=0)
        cov = np.cov(cluster_points, rowvar=False)

        if cov.ndim == 0:
            cov = np.array([[cov]])
        elif cov.ndim == 1:
            cov = np.diag(cov)

        cov += np.eye(cov.shape[0]) * 1e-6  # regularização

        try:
            synthetic = np.random.multivariate_normal(centroid, cov, samples_per_cluster)
            synthetic_samples.append(synthetic)
        except:
            continue  # ignora se a covariância for inválida

    if synthetic_samples:
        return np.vstack(synthetic_samples)
    else:
        return np.empty((0, X_class.shape[1]))  # sem amostras geradas

## For para aumentar os agrupamentos presentes nos dicionários:

In [38]:
synthetic_samples_list = []
synthetic_labels_list = []

for label, group in dict_groups.items():
    features = group.drop(columns='act').values
    synthetic_data = mean_shift_generator(features, target_total=70)

    if len(synthetic_data) == 0:
        continue

    df_synthetic = pd.DataFrame(synthetic_data, columns=['h2','ch2','c2h2','c2h4','c2h6'])
    synthetic_samples_list.append(df_synthetic)
    synthetic_labels_list.extend([label - 1] * len(df_synthetic))



## Combina os DataFrames e junta as samples e labels:

In [39]:
augmented_samples_df = pd.concat(synthetic_samples_list, ignore_index=True)
augmented_labels_df = pd.DataFrame(synthetic_labels_list, columns=['act'])

final_df = pd.concat([augmented_samples_df, augmented_labels_df], axis=1)

## Escreve os arquivos .csv:

In [40]:
filename = "Datasets/Ibrahim-dataset/MS-oversampled-dataset/entire_dataset.csv"

final_df.to_csv(filename, index=False)